In [1]:
"""B64 two-point analysis: standard multistate fits, GEVP comparison, and Laplace checks."""

from pathlib import Path
import os
import sys

import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
if HERE.parent.name == "__codex_ignore":
    HERE = HERE.parent.parent / HERE.name
if HERE.name != "cB211.072.64" or HERE.parent.name != "07_Nsgm":
    raise RuntimeError("Launch this notebook from its cB211.072.64 directory.")
WORK = HERE.parent / "__codex_ignore" / HERE.name
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(HERE.parent))

import util as yu
import util_codex as yuc

yu.setpath("analysis_2pt_codex")
ENS = "b"
A = yu.ens2a[ENS]
AINV_GEV = yu.ens2aInv[ENS] / 1000
DELTA = 2
SELECTED_METHOD = {"N": 0, "Nsgm": 1}
FIT_SELECTION = {"N": [(20, 8, 3), (20, 8, 3)], "Nsgm": [(13, 4), (13, 4)]}


# Analysis helpers
The helpers below implement only the two-point transformations and fits.


In [2]:
def fit_constant_scans(data, lower_bounds, upper_bound):
    return yu.doFits_const(data, lower_bounds, [upper_bound], corrQ=False)


In [3]:
def selected_fit(fits, label):
    return next(fit for fit in fits if fit[0] == label)


In [4]:
def model_average_index(fits):
    _, probabilities = yu.jackMA(fits)
    return int(np.argmax(np.mean(probabilities, axis=0)))


In [5]:
def project_correlator(correlator, vector):
    return np.real(np.einsum("ni,ntij,nj->nt", vector, correlator, vector))


In [6]:
def fit_multistate_effective_masses(correlators):
    ranges = {
        "N": [range(2, 28), range(2, 13), range(1, 5)],
        "Nsgm": [range(2, 16), range(2, 9)],
    }
    starts = {"N": [.4, .5, 2, .8, 1], "Nsgm": [.52, .25, 1, .35, 1]}
    effective_masses = {
        key: tuple(yu.jackmap(yu.c2pt2meff, correlator) for correlator in values)
        for key, values in correlators.items()
    }
    fits = {}
    for key, masses in effective_masses.items():
        fits[key] = tuple(
            yuc.fit_meff_comparison(
                mass, ranges[key], starts[key].copy(), corrQ=True,
                label=f"{key}_{tag}_dense_codex_v3", overwrite=False,
            )
            for mass, tag in zip(masses, ["std", "GEVP"])
        )
    return effective_masses, fits


In [7]:
def laplace_scan(correlator, initial_fit, channel):
    energy0, gap = np.mean(initial_fit[:, :2], axis=0)
    energy1 = energy0 + gap
    low, high = ((.48, 1.5) if channel == "N" else (.7, 1.2))
    midpoint, half_width = (low + high) / 2, (high - low) / 2
    to_energy = lambda q: midpoint + half_width * np.tanh(q)
    to_parameter = lambda energy: np.arctanh((energy - midpoint) / half_width)

    def filtered(energy):
        return 2 * np.cosh(DELTA * energy) * correlator[:, DELTA:-DELTA] \
            - correlator[:, 2 * DELTA:] - correlator[:, :-2 * DELTA]

    def effective_mass(parameter):
        data = filtered(to_energy(parameter))
        return np.log(data / np.roll(data, -1, axis=1))

    mean, error = yu.jackme(effective_mass(to_parameter(energy1)))
    relative_error = np.abs(error / mean)
    stop = next(
        (index for index in range(8, len(mean) - 1)
         if np.all((~np.isfinite(relative_error[index:index + 2])) | (relative_error[index:index + 2] > .35))),
        len(mean) - 1,
    )
    upper_bound = DELTA + stop

    def fit(lower_bound):
        times = np.arange(lower_bound, upper_bound)
        parameters, chi2, ndof, _ = yu.jackfit(
            lambda pars: np.full(len(times), pars[0]),
            lambda q: effective_mass(q)[:, times - DELTA],
            [energy0, to_parameter(energy1)], maxfev=2000,
        )
        parameters[:, 1] = to_energy(parameters[:, 1]) - parameters[:, 0]
        return lower_bound, parameters, chi2, ndof

    last_start = 12 if channel == "N" else 11
    fits = [fit(lower) for lower in range(4, last_start) if lower < upper_bound - 1]
    return fits, filtered, upper_bound


In [8]:
def fit_displacement(correlator, initial_fit, channel, displacement):
    energy0, gap = np.mean(initial_fit[:, :2], axis=0)
    energy1 = energy0 + gap
    low, high = ((.48, 1.5) if channel == "N" else (.7, 1.2))
    midpoint, half_width = (low + high) / 2, (high - low) / 2
    to_energy = lambda q: midpoint + half_width * np.tanh(q)
    to_parameter = lambda energy: np.arctanh((energy - midpoint) / half_width)
    # Fix the earliest and latest original correlator times for every displacement.
    first, last = {"N": (7, 28), "Nsgm": (4, 16)}[channel]
    times = np.arange(first, last - 2 * displacement)
    if len(times) <= 2:
        raise ValueError("The two-parameter fit needs at least three effective masses.")

    def effective_mass(parameter):
        energy = to_energy(parameter)
        data = 2 * np.cosh(displacement * energy) * correlator[:, displacement:-displacement]
        data -= correlator[:, 2 * displacement:] + correlator[:, :-2 * displacement]
        return np.log(data / np.roll(data, -1, axis=1))

    parameters, chi2, ndof, _ = yu.jackfit(
        lambda pars: np.full(len(times), pars[0]),
        lambda q: effective_mass(q)[:, times],
        [energy0, to_parameter(energy1)], maxfev=2000,
    )
    parameters[:, 1] = to_energy(parameters[:, 1]) - parameters[:, 0]
    return parameters, chi2, ndof


# Analysis
Determine the eigenvector ratios first, then analyze standard and projected correlators.


In [9]:
c2pt_matrix, _, _ = yu.load_pkl(HERE / "pkl/processData/reg_ignore/data.pkl")
reference_times = np.arange(20)
times = reference_times + 2
eigenvectors = np.array([yu.GEVP(correlator, reference_times, tList=times)[1]
                         for correlator in c2pt_matrix])
inverse_eigenvectors = np.linalg.inv(eigenvectors)

r0_series = np.real(eigenvectors[:, :, 0, 1]) / np.real(eigenvectors[:, :, 0, 0])
s1_series = np.real(eigenvectors[:, :, 1, 0]) / np.real(eigenvectors[:, :, 1, 1])
w_series = 1 / (np.real(eigenvectors[:, :, 0, 0]) * np.real(inverse_eigenvectors[:, :, 0, 0])) - 1
component_series = [r0_series, s1_series, w_series]
component_lower_bounds = np.arange(1, 11)
component_fits = [fit_constant_scans(data, component_lower_bounds, 12) for data in component_series]
component_selected = [selected_fit(fits, (7, 12))[1][:, 0] for fits in component_fits]
w_reconstructed_fits = [-fit_r[1][:, 0] * fit_s[1][:, 0]
                        for fit_r, fit_s in zip(component_fits[0], component_fits[1])]
w_reconstructed = w_reconstructed_fits[list(component_lower_bounds).index(7)]
r0, s1 = component_selected[:2]
yu.save_pkl_reg("evec_ratios", [r0, s1, w_reconstructed])

yuc.guard_fit_cache(c2pt_matrix, r0, s1, w_reconstructed)


## Standard and projected two-point fits


In [10]:
vector0 = np.stack([np.ones_like(r0), r0], axis=1)
vector1 = np.stack([s1, np.ones_like(s1)], axis=1)
correlators = {
    "N": (np.real(c2pt_matrix[:, :, 0, 0]), project_correlator(c2pt_matrix, vector0)),
    "Nsgm": (np.real(c2pt_matrix[:, :, 1, 1]), project_correlator(c2pt_matrix, vector1)),
}
effective_masses, multistate_fits = fit_multistate_effective_masses(correlators)
standard_selected = {
    channel: selected_fit(multistate_fits[channel][0][1], FIT_SELECTION[channel][0][1])[1]
    for channel in correlators
}
yu.save_pkl_reg("standard_two_state_selected", standard_selected)
two_state_selected = {
    channel: selected_fit(multistate_fits[channel][method][1], FIT_SELECTION[channel][method][1])[1]
    for channel, method in SELECTED_METHOD.items()
}
yu.save_pkl_reg("two_state_selected", two_state_selected)


## Laplace fits


In [11]:
laplace = {}
laplace_selection = {"N": 10, "Nsgm": 6}
laplace_selected = {}
laplace_effective_masses = {}
for channel in correlators:
    method = SELECTED_METHOD[channel]
    two_state_fits = multistate_fits[channel][method][1]
    initial = two_state_fits[model_average_index(two_state_fits)][1]
    laplace[channel] = laplace_scan(correlators[channel][method], initial, channel)
    laplace_selected[channel] = selected_fit(laplace[channel][0], laplace_selection[channel])[1]
    laplace_effective_masses[channel] = yu.jackmap(
        yu.c2pt2meff,
        np.array([
            2 * np.cosh(DELTA * np.sum(pars)) * correlator[DELTA:-DELTA]
            - correlator[2 * DELTA:] - correlator[:-2 * DELTA]
            for correlator, pars in zip(correlators[channel][method], laplace_selected[channel])
        ]),
    )
yu.save_pkl_reg("laplace_selected", laplace_selected)


## Discretization checks


In [12]:
displacement_ranges = {"N": range(1, 5), "Nsgm": range(1, 5)}
displacement_fits = {}
for channel in correlators:
    method = SELECTED_METHOD[channel]
    two_state_fits = multistate_fits[channel][method][1]
    initial = two_state_fits[model_average_index(two_state_fits)][1]
    displacement_fits[channel] = {
        displacement: fit_displacement(correlators[channel][method], initial, channel, displacement)
        for displacement in displacement_ranges[channel]
    }
yu.save_pkl_reg("laplace_delta_fits", displacement_fits)

print("v0Ns/v0N, v1N/v1Ns, direct W, reconstructed W:",
      *[yu.jackme_un2str(value) for value in [r0, s1, component_selected[2], w_reconstructed]])
for channel in correlators:
    print(f"\n{channel} multistate fits")
    for method, fits_by_state in zip(["standard", "GEVP"], multistate_fits[channel]):
        values = []
        for state_count, fits in enumerate(fits_by_state, start=1):
            fit = fits[model_average_index(fits)]
            values.append((state_count, fit[0], yu.jackme_un2str(fit[1][:, 0] * AINV_GEV)))
        print(method, values)


v0Ns/v0N, v1N/v1Ns, direct W, reconstructed W: 0.01563(99) -0.550(17) 0.00851(77) 0.00860(78)

N multistate fits
standard [(1, 16, '0.9537(24)'), (2, 8, '0.9440(44)'), (3, 4, '0.933(14)')]
GEVP [(1, 16, '0.9531(28)'), (2, 8, '0.9426(52)'), (3, 4, '0.934(15)')]

Nsgm multistate fits
standard [(1, 9, '1.348(16)'), (2, 4, '1.300(25)')]
GEVP [(1, 9, '1.361(17)'), (2, 4, '1.318(25)')]


# Plotting helpers
Figure construction is isolated below the analysis and does not alter selected quantities.


In [13]:
PLOT_STYLE = yuc.paper_style({"lines.markersize": 3.3, "errorbar.capsize": 2.5})

def plot_eigenvectors():
    components = dict(zip(["v", "s", "w"], component_series))
    scans = {name: [(times[lower], fit[1][:, 0], fit[2], fit[3])
                    for lower, fit in zip(component_lower_bounds, fits)]
             for name, fits in zip(components, component_fits)}
    config = {"window": (9, 14), "display": (3, 17),
              "rows": [dict(ylim=(-.01, .05), yticks=[0, .02, .04]),
                       dict(ylim=(-.72, .05), yticks=[-.6, -.4, -.2, 0]),
                       dict(ylim=(-.008, .028), yticks=[0, .01, .02])],
              "left": dict(xlim=(0, 1.5), xticks=[0, .5, 1]),
              "right": dict(xlim=(.15, 1.08), xticks=[.2, .4, .6, .8, 1])}
    yuc.plot_eigenvectors(times, components, scans, dict(zip(components, component_selected)), A, config)


In [14]:
def plot_correlator_comparison(channel, include_three_state):
    stop = {"N": 30, "Nsgm": 17}[channel]
    config = {"display_times": [np.arange(1, stop + 1)] * 2, "include_three_state": include_three_state}
    if channel == "N":
        config.update(effective=dict(xlim=(0, 2.5), ylim=(.75, 1.4)),
                      ground=dict(ylim=(.89, 1.15)), excited=dict(ylim=(.7, 3.5)))
    else:
        config.update(effective=dict(xlim=(0, 1.5), ylim=(.45, 2.65)),
                      ground=dict(ylim=(.8, 2)), excited=dict(ylim=(0, 4.6)))
    stop_lap = laplace[channel][2] + (1 if channel == "Nsgm" else 0)
    t = np.arange(DELTA + 2, stop_lap) - DELTA
    lap = (t, laplace_effective_masses[channel][:, t],
           [(f[0] - DELTA, *f[1:]) for f in laplace[channel][0]])
    selection = (SELECTED_METHOD[channel], FIT_SELECTION[channel][SELECTED_METHOD[channel]][1])
    yuc.plot_correlator_comparison(channel, effective_masses[channel], multistate_fits[channel],
                                  selection, two_state_selected[channel], lap, A, AINV_GEV, config)


In [15]:
def plot_displacement_dependence():
    with mpl.rc_context(PLOT_STYLE):
        fig, axis = plt.subplots(figsize=(3.4, 2.65))
        categories = [("N", 0, r"$E_0^N$"), ("N", 1, r"$E_1^N$"),
                      ("Nsgm", 0, r"$\widetilde E_0^{N\sigma}$"), ("Nsgm", 1, r"$\widetilde E_1^{N\sigma}$")]
        displayed = {"N": range(1, 5), "Nsgm": range(1, 5)}
        colors = [yu.colors8[i] for i in [2, 3, 1, 0]]
        markers = ["o", "s", "^", "d"]
        for displacement, color, marker in zip(range(1, 5), colors, markers):
            positions, samples = [], []
            for position, (channel, index, _) in enumerate(categories):
                if displacement not in displayed[channel]:
                    continue
                count = len(displayed[channel])
                positions.append(position + .075 * (displacement - (count + 1) / 2))
                parameters = displacement_fits[channel][displacement][0]
                samples.append(parameters[:, 0] if index == 0 else np.sum(parameters, axis=1))
            values = np.array([yu.jackme(sample * AINV_GEV) for sample in samples])
            yuc.errorbar_with_selected(
                axis, positions, values[:, 0], values[:, 1],
                np.ones(len(positions), dtype=bool) if displacement == 2 else None,
                fmt=marker, color=color, label=str(displacement))
        axis.legend(loc="upper left", ncols=4, fontsize=7, title=r"$\delta/a$", title_fontsize=7,
                    columnspacing=.9, handletextpad=.25)
        axis.set(ylabel=r"$E(\delta)$ [GeV]", xlim=(-.4, 3.4), ylim=(.85, 3.3),
                 xticks=np.arange(len(categories)), xticklabels=[label for _, _, label in categories])
        fig.tight_layout()
        yu.finalizePlot("C2pt_laplace_delta_dependence", tightQ=False)


In [16]:
def plot_overlap_ratios():
    scans = {(channel, state): multistate_fits[channel][SELECTED_METHOD[channel]][state]
             for channel, state in [("N", 1), ("N", 2), ("Nsgm", 1)]}
    selections = {c: FIT_SELECTION[c][SELECTED_METHOD[c]][1] for c in ["N", "Nsgm"]}
    yuc.plot_overlap_ratios(scans, selections, A,
                           dict(xlim=(.05, 1.08), ylim=(0, 6.8), yticks=np.arange(0, 6.1)))


# Plotting
Generate the two-point figures after all selected analysis quantities are fixed.


In [17]:
plot_eigenvectors()
plot_correlator_comparison("N", include_three_state=True)
plot_correlator_comparison("Nsgm", include_three_state=False)
plot_displacement_dependence()
plot_overlap_ratios()
